In [ ]:
import numpy as np
from scipy import stats

variant = 1
july_table = 21.0  # липнева температура Києва з таблиці

np.random.seed(variant)
daily_temps = np.random.normal(loc=july_table, scale=2.5, size=30)
np.random.seed()

print(daily_temps)
print("mean =", daily_temps.mean())
print("std(ddof=1) =", daily_temps.std(ddof=1))

In [ ]:
n = len(daily_temps)
mean = daily_temps.mean()
s = daily_temps.std(ddof=1)
se = s / np.sqrt(n)

# 95% довірчий інтервал для середнього (t-розподіл)
ci_mean_95 = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)

print("x̄ = {:.4f}".format(mean))
print("s (ddof=1) = {:.4f}".format(s))
print("se = {:.4f}".format(se))
print("95% CI для середнього: ({:.4f}, {:.4f})".format(ci_mean_95[0], ci_mean_95[1]))

In [ ]:
# частка днів, коли temp > 21
above_july = daily_temps > july_table
p_hat = above_july.mean()
n_val = len(daily_temps)

n_p = n_val * p_hat
n_1p = n_val * (1 - p_hat)

print("Кількість днів з temp > {:.0f}: {}".format(july_table, int(above_july.sum())))
print("p_hat = {:.4f}".format(p_hat))
print("n * p_hat = {:.1f}".format(n_p))
print("n * (1 - p_hat) = {:.1f}".format(n_1p))

# перевірка умови нормального наближення
if n_p >= 5 and n_1p >= 5:
    print("Умова n*p >= 5 і n*(1-p) >= 5 ВИКОНАНА — наближення прийнятне.")
else:
    print("Умова НЕ виконана — наближення ненадійне.")

# 95% CI для частки
se_p = np.sqrt(p_hat * (1 - p_hat) / n_val)
ci_prop = stats.norm.interval(0.95, loc=p_hat, scale=se_p)

print("95% CI для частки: ({:.4f}, {:.4f})".format(ci_prop[0], ci_prop[1]))
print("Чи потрапляє 0.5 в інтервал? {}".format(ci_prop[0] <= 0.5 <= ci_prop[1]))

In [ ]:
results = []
for conf in [0.90, 0.95, 0.99]:
    ci = stats.t.interval(conf, df=n - 1, loc=mean, scale=se)
    width = ci[1] - ci[0]
    results.append((conf, ci[0], ci[1], width))

print("{:>8} | {:>10} | {:>10} | {:>10}".format("рівень", "нижня", "верхня", "ширина"))
print("-" * 46)
for conf, low, high, w in results:
    print("{:>8}% | {:>10.4f} | {:>10.4f} | {:>10.4f}".format(int(conf * 100), low, high, w))

## Контрольні питання

### 1. Точкова оцінка vs довірчий інтервал

Точкова оцінка (наприклад, x̄ = 21.3 °C) — це одне число, «найкраща здогадка» про справжнє значення. Але вона не дає ніякої інформації про **невизначеність** цієї здогадки: скільки ми можемо помилятися? Довірчий інтервал додає до цього діапазон «наскільки може бути інакше» — він показує, в межах якого діапазону лежить справжнє значення з заданим рівнем впевненості. Тобто інтервал відповідає на питання «з якою точністю?» — точкова оцінка — лише «як приблизно?».

### 2. Чому t-розподіл, а не нормальний

t-розподіл використовується, коли **σ невідоме** і оцінюється за вибіркою (s). При цьому додається ще одна джерело невизначеності — невизначеність самої оцінки s. t-розподіл має довші хвости, ніж нормальний, що враховує цю додаткову невизначеність. Коли n велике (зазвичай n ≥ 30–50), t ≈ z, але при n = 30 різниця помітна: t_0.95,29 ≈ 2.045 замість z_0.95 = 1.960.

### 3. Що означає "95% довіри"

«95% довіри» означає, що **метод побудови інтервалів** при повторенні експерименту дає інтервали, які накривають справжнє значення у 95% випадків. Це не ймовірність для конкретного інтервалу — після того як інтервал обчислено, він або накриває справжнє значення (ймовірність = 1), або ні (ймовірність = 0). «95%» — це властивість процесу, а не результату.

Поширена помилка «з ймовірністю 95% справжня середня лежить у цьому інтервалі» неправильна, бо справжня середня — фіксоване число, а не випадкова величина. Вона не «рухається» і не має ймовірності опинитися десь.

### 4. Три фактори ширини довірчого інтервалу

Ширина CI = 2 × крит_значення × se, де se = s/√n. Три фактори:

1. **s (вибіркове стандартне відхилення)** — чим більший розкид даних, тим ширший інтервал. Це властивість самих даних.
2. **n (розмір вибірки)** — чим більше спостережень, тим вужчий інтервал (se ∝ 1/√n). Для подвоєння точності потрібно вчетверо більше даних.
3. **Рівень довіри** — вищий рівень довіри → більше критичне значення → ширший інтервал. Обмін: більше впевненість = менше точність.